In [0]:
# CELL 1: Parameters and Filtering
dbutils.widgets.text("batch_id", "incremental_run")
batch_id = dbutils.widgets.get("batch_id")

from pyspark.sql.functions import col, to_date, concat_ws, lit, current_timestamp, lpad
from delta.tables import DeltaTable
from datetime import datetime
start_time = datetime.now()

df_bronze = spark.read.table("default.bronze_wfp_raw")

# Focus exclusively on Pakistan per business requirements
df_filtered = df_bronze.filter(col("adm0_name") == "Pakistan")

In [0]:
# CELL 2: Transformation
df_silver_stage = df_filtered.select(
    col("adm0_name").alias("country"),
    col("adm1_name").alias("admin_region"),
    col("mkt_name").alias("market"),
    col("cm_name").alias("commodity_name"),
    col("um_name").alias("unit"),
    col("mp_price").cast("double").alias("price"),
    to_date(concat_ws("-", col("mp_year"), lpad(col("mp_month"), 2, "0"), lit("01"))).alias("price_date")
).withColumn("is_revised_flag", lit(False)) \
 .withColumn("load_timestamp", current_timestamp())

print("--- Pakistan Specific WFP Data ---")
display(df_silver_stage.limit(5))

--- Pakistan Specific WFP Data ---


country,admin_region,market,commodity_name,unit,price,price_date,is_revised_flag,load_timestamp
Pakistan,Balochistan,Quetta,Wheat flour - Retail,KG,13.0,2004-01-01,false,2026-10-07T18:56:29.427Z
Pakistan,Balochistan,Quetta,Wheat flour - Retail,KG,13.0,2004-02-01,false,2026-10-07T18:56:29.427Z
Pakistan,Balochistan,Quetta,Wheat flour - Retail,KG,14.25,2004-03-01,false,2026-10-07T18:56:29.427Z
Pakistan,Balochistan,Quetta,Wheat flour - Retail,KG,12.5,2004-04-01,false,2026-10-07T18:56:29.427Z
Pakistan,Balochistan,Quetta,Wheat flour - Retail,KG,13.25,2004-05-01,false,2026-10-07T18:56:29.427Z


In [0]:
# CELL 3: Upsert Logic (Idempotent Merge)
import uuid as _uuid

# Add columns expected by the target table schema
df_silver_stage = df_silver_stage.withColumn("category", lit("Unknown")) \
    .withColumn("currency", lit("PKR")) \
    .withColumn("ingestion_batch_id", lit(batch_id))

try:
    if not spark.catalog.tableExists("default.silver_wfp_price_observation"):
        df_silver_stage.write.format("delta").saveAsTable("default.silver_wfp_price_observation")
        rows = df_silver_stage.count()
    else:
        silver_table = DeltaTable.forName(spark, "default.silver_wfp_price_observation")
        
        # Match on primary keys
        silver_table.alias("target").merge(
            df_silver_stage.alias("source"),
            "target.market = source.market AND target.commodity_name = source.commodity_name AND target.price_date = source.price_date"
        ).whenMatchedUpdate(
            condition="target.price != source.price",
            set={"price": "source.price", "is_revised_flag": "true", "load_timestamp": "source.load_timestamp"}
        ).whenNotMatchedInsertAll().execute()
        rows = df_silver_stage.count()

    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Bronze-to-Silver-WFP', '{batch_id}', '{start_time}', current_timestamp(), 'Success', {rows}, '')")
except Exception as e:
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Bronze-to-Silver-WFP', '{batch_id}', '{start_time}', current_timestamp(), 'Failure', 0, '{str(e)[0:200]}')")
    raise e